## Identify next batch to process

This notebook determines the next available `batch_id` to process by comparing:

* Batch folders present in the landing volume at `/Volumes/company_db/landing/files`
* Batch IDs already recorded in `company_db.batch_control.batch_table`

It performs the following steps:

* Lists available batch folders from the landing location
* Reads processed batch IDs from the batch control table
* Finds the difference between available and processed batches
* Selects the earliest remaining batch as the next batch to process
* Sets Databricks job task values:
  * `id` = next batch ID
  * `has_batch` = `true` or `false`

This notebook is intended to be used in an orchestration flow where downstream tasks read the task values to decide whether a batch should be processed.

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
# List batch folders currently available in the landing volume.
# Each folder name is treated as a candidate batch_id.
folder_path = dbutils.fs.ls(f'/Volumes/{catalog}/landing/files')
landing_batches = [i.name.split('/')[0] for i in folder_path]
landing_batches = sorted(landing_batches)

print(f"Files in ADLS: {landing_batches}")

# Read the batch IDs that have already been registered in the control table.
# These are considered already processed or already tracked by the pipeline.
df = spark.sql(f"""
    select distinct(batch_id)
    from {catalog}.batch_control.batch_table
    WHERE  batch_status = 'completed' or batch_status is null 
""")

# Convert the Spark DataFrame results into a Python list for comparison.
# If nothing has been tracked yet, default to an empty list.
tracked_batches = [i.batch_id for i in df.collect()]
if not tracked_batches:
    tracked_batches = []

print(f"Batches in batch_control(Already processed): {tracked_batches}")

# Compare landing batches with tracked batches to find the remaining batches.
# The earliest batch in sorted order becomes the next batch to process.
next_batches = sorted(list(set(landing_batches) - set(tracked_batches)))
next_batch = next_batches[0] if next_batches else None

print(f"Next batch to process: {next_batch}")

# # Publish the result as job task values so downstream tasks can consume it.
# # If no batch is available, set an empty id and mark has_batch as false.
if next_batch is None:
    dbutils.jobs.taskValues.set(key='id', value="")
    dbutils.jobs.taskValues.set(key='has_batch', value="false")
else:
    dbutils.jobs.taskValues.set(key='id', value=next_batch)
    dbutils.jobs.taskValues.set(key='has_batch', value="true")
